# RegimeLab — EURUSD Feature Intelligence
Information Coefficient (IC), sign consistency, drift and redundancy analysis.


In [ ]:
# Papermill Parameter Contract
import os
from pathlib import Path
WORKSPACE_ID = ''
EXPERIMENT_ID = ''
ARTIFACT_DIR = os.environ.get('ARTIFACT_DIR', './artifacts')
RUN_ID = os.environ.get('RUN_ID', 'manual')
LOOKBACK_BARS = 60
FORWARD_BARS = 1
Path(ARTIFACT_DIR).mkdir(parents=True, exist_ok=True)


In [ ]:
import pandas as pd
import numpy as np
from backend.engine import demo_prices, features
from regimelab_sdk import run

df = demo_prices(600)
feat_df = features(df)
returns = df['close'].pct_change(FORWARD_BARS).shift(-FORWARD_BARS)

ic_metrics = {}
for col in feat_df.columns:
    if col in ['close', 'open', 'high', 'low']:
        continue
    corr = feat_df[col].corr(returns)
    if not np.isnan(corr):
        ic_metrics[col] = float(round(corr, 4))

top_feature = max(ic_metrics, key=lambda k: abs(ic_metrics[k]))
max_ic = ic_metrics[top_feature]
print(f'Top feature: {top_feature} (IC = {max_ic})')

run.log_metric('top_ic', abs(max_ic))
run.log_metric('features_analyzed', len(ic_metrics))
ic_path = Path(ARTIFACT_DIR) / 'feature_ic_report.csv'
pd.DataFrame([{'feature': k, 'ic': v} for k, v in ic_metrics.items()]).to_csv(ic_path, index=False)
run.log_artifact(ic_path)
run.log_message('Feature Intelligence analysis completed.')
